<a href="https://colab.research.google.com/github/prithamraja/MoRD_LokOS/blob/main/DB_Generate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
LokOS Synthetic Database Generator
Medium scale: 3 states, ~20 districts, ~5,000 SHGs
All 4 layers: Profile, SHG Transactions, CLF Transactions, Digital Aajeevika Register
"""

import sqlite3
import random
import math
from datetime import date, timedelta, datetime
import hashlib
import os

random.seed(42)
DB_PATH = "/mnt/user-data/outputs/lokos_synthetic.db"

# ─── Reference data ──────────────────────────────────────────────────────────

STATES = [
    {"id": "JH", "name": "Jharkhand",     "srlm": "JSLPS"},
    {"id": "MP", "name": "Madhya Pradesh", "srlm": "MPSRLM"},
    {"id": "OD", "name": "Odisha",         "srlm": "OSORLM"},
]

DISTRICTS = {
    "JH": ["Ranchi", "Dumka", "Giridih", "Hazaribagh", "Bokaro", "Dhanbad", "Palamu"],
    "MP": ["Mandla", "Balaghat", "Dindori", "Seoni", "Chhindwara", "Jabalpur", "Narsinghpur"],
    "OD": ["Koraput", "Rayagada", "Kandhamal", "Kalahandi", "Gajapati", "Malkangiri", "Nabarangpur"],
}

# Blocks per district (we'll generate names programmatically)
BLOCKS_PER_DISTRICT = 4
VOS_PER_BLOCK = 8          # ~3–4 CLFs per block, ~2–3 VOs per CLF → ~8 VOs/block
SHGS_PER_VO = 7            # 5–10 SHGs per VO
MEMBERS_PER_SHG = (10, 14) # 10–14 members

FEMALE_FIRST_NAMES = [
    "Sunita","Meena","Geeta","Parvati","Savitri","Laxmi","Durga","Kamla","Rekha","Usha",
    "Shanti","Pushpa","Kiran","Nirmala","Saroj","Anita","Rita","Sita","Mala","Vandana",
    "Rohini","Seema","Asha","Priya","Radha","Gita","Mamta","Kavita","Urmila","Bindiya",
    "Sushma","Chanda","Bimla","Renu","Santoshi","Jamuna","Shakuntala","Hemlata","Sudha",
    "Sangita","Chandrakala","Phoolmati","Rukmini","Draupadi","Chandrawati","Saraswati",
    "Yashodha","Tulsi","Indira","Champa"
]

SURNAMES = [
    "Devi","Kumari","Bai","Nag","Oraon","Munda","Sahu","Patel","Yadav","Sharma",
    "Gupta","Singh","Mahato","Gond","Bhil","Majhi","Minj","Tigga","Kujur","Ekka",
    "Lakra","Xalxo","Kandulna","Toppo","Barla","Horo","Soreng","Tirkey","Tete",
    "Kerketta","Kandel","Biswal","Panda","Pradhan","Nayak","Behera","Dash","Parida"
]

BANK_NAMES = [
    "State Bank of India","Bank of Baroda","Punjab National Bank","Union Bank of India",
    "Canara Bank","Indian Bank","Central Bank of India","UCO Bank",
    "Jharkhand Rajya Gramin Bank","Madhya Pradesh Gramin Bank","Utkal Grameen Bank",
    "HDFC Bank","Axis Bank"
]

LOAN_CATEGORIES = ["Consumption","Agriculture","Livestock & Fisheries","NTFP","Non-Farm"]
LOAN_PURPOSES = {
    "Consumption":           ["Medical emergency","School fees","Food purchase","House repair","Wedding expenses"],
    "Agriculture":           ["Seeds and fertilizer","Land preparation","Irrigation","Crop inputs","Equipment repair"],
    "Livestock & Fisheries": ["Purchase of goat","Purchase of cow","Poultry inputs","Fishnet purchase","Fodder"],
    "NTFP":                  ["Tendu leaf collection","Mahua collection","Sal seed processing","Bamboo craft materials","Forest produce"],
    "Non-Farm":              ["Tailoring materials","Petty trade","Kirana shop stock","Pottery inputs","Masonry tools"],
}

LIVELIHOOD_ACTIVITIES = [
    "Animal Husbandry","Aqua Farming","Crop Based","NTFP",
    "Manufacturing","Trading","Services","Daily Wages","Skill Based"
]

FUND_TYPES = ["CIF","VRF","CEF","AGEY-CIF","PMFME-Seed Capital","Women Enterprise Acceleration Fund"]

RECEIPT_TYPES = ["Share Capital","Other Savings","Membership Fees","Penalty"]
PAYMENT_TYPES = ["Grant/Donation","Dividend Payment"]
MODES = ["Cash","Bank","Online"]

REGISTRATION_ACTS = [
    "State Cooperative Societies Act","Societies Registration Act",
    "Multi-State Cooperative Societies Act","Public Trust Act"
]

# ─── Utility helpers ─────────────────────────────────────────────────────────

def uid(prefix, *parts):
    raw = prefix + "_" + "_".join(str(p) for p in parts)
    return raw.replace(" ", "_")

def fake_aadhaar():
    return str(random.randint(2, 9)) + "".join(str(random.randint(0,9)) for _ in range(11))

def fake_phone():
    return "9" + "".join(str(random.randint(0,9)) for _ in range(9))

def fake_ifsc(bank_name):
    code = ''.join(c for c in bank_name.upper() if c.isalpha())[:4].ljust(4,'X')
    return code + "0" + str(random.randint(100000, 999999))

def fake_account():
    return str(random.randint(10**10, 10**14))

def rand_date(start, end):
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, delta))

def date_str(d):
    return d.strftime("%Y-%m-%d")

def name():
    return random.choice(FEMALE_FIRST_NAMES) + " " + random.choice(SURNAMES)

# ─── Schema creation ─────────────────────────────────────────────────────────

SCHEMA = """
-- GEOGRAPHY
CREATE TABLE IF NOT EXISTS states (
    state_id TEXT PRIMARY KEY, state_name TEXT, srlm_name TEXT
);
CREATE TABLE IF NOT EXISTS districts (
    district_id TEXT PRIMARY KEY, district_name TEXT, state_id TEXT,
    FOREIGN KEY(state_id) REFERENCES states(state_id)
);
CREATE TABLE IF NOT EXISTS blocks (
    block_id TEXT PRIMARY KEY, block_name TEXT, district_id TEXT,
    FOREIGN KEY(district_id) REFERENCES districts(district_id)
);
CREATE TABLE IF NOT EXISTS gram_panchayats (
    gp_id TEXT PRIMARY KEY, gp_name TEXT, block_id TEXT,
    FOREIGN KEY(block_id) REFERENCES blocks(block_id)
);

-- COMMUNITY ORGANISATIONS
CREATE TABLE IF NOT EXISTS clfs (
    clf_id TEXT PRIMARY KEY, clf_name TEXT, block_id TEXT,
    formation_date TEXT, registration_number TEXT, registration_act TEXT,
    date_of_registration TEXT, status TEXT, meeting_frequency TEXT,
    financial_intermediation TEXT, member_organisation TEXT,
    compulsory_saving_freq TEXT, compulsory_saving_amt REAL,
    compulsory_saving_rate REAL, bookkeeper_name TEXT, bookkeeper_phone TEXT,
    bank_name TEXT, bank_account TEXT, ifsc TEXT, bank_balance REAL,
    FOREIGN KEY(block_id) REFERENCES blocks(block_id)
);
CREATE TABLE IF NOT EXISTS vos (
    vo_id TEXT PRIMARY KEY, vo_name TEXT, gp_id TEXT, clf_id TEXT,
    formation_date TEXT, status TEXT, meeting_frequency TEXT,
    bookkeeper_name TEXT, bookkeeper_phone TEXT,
    bank_name TEXT, bank_account TEXT, ifsc TEXT, bank_balance REAL,
    FOREIGN KEY(gp_id) REFERENCES gram_panchayats(gp_id),
    FOREIGN KEY(clf_id) REFERENCES clfs(clf_id)
);
CREATE TABLE IF NOT EXISTS shgs (
    shg_id TEXT PRIMARY KEY, shg_name TEXT, vo_id TEXT,
    formation_date TEXT, status TEXT, promoted_by TEXT,
    meeting_frequency TEXT, compulsory_saving_amt REAL,
    compulsory_saving_rate REAL, voluntary_saving TEXT,
    bookkeeper_name TEXT, bookkeeper_phone TEXT,
    bank_name TEXT, bank_account TEXT, ifsc TEXT, bank_balance REAL,
    revolving_fund_recd REAL, cif_recd REAL, total_corpus REAL,
    panchsutra_score INTEGER,
    FOREIGN KEY(vo_id) REFERENCES vos(vo_id)
);
CREATE TABLE IF NOT EXISTS members (
    member_id TEXT PRIMARY KEY, shg_id TEXT, member_name TEXT,
    aadhaar TEXT UNIQUE, mobile TEXT, dob TEXT,
    caste_category TEXT, bpl_status TEXT,
    is_lakhpati_didi INTEGER DEFAULT 0,
    joining_date TEXT, is_office_bearer INTEGER,
    office_bearer_role TEXT,
    FOREIGN KEY(shg_id) REFERENCES shgs(shg_id)
);

-- SHG TRANSACTIONS
CREATE TABLE IF NOT EXISTS shg_meetings (
    meeting_id TEXT PRIMARY KEY, shg_id TEXT, meeting_date TEXT,
    total_members INTEGER, members_present INTEGER,
    total_savings_collected REAL, total_loan_repayments REAL,
    total_loans_disbursed REAL, cash_in_hand REAL,
    FOREIGN KEY(shg_id) REFERENCES shgs(shg_id)
);
CREATE TABLE IF NOT EXISTS member_savings (
    savings_id TEXT PRIMARY KEY, meeting_id TEXT, shg_id TEXT,
    member_id TEXT, savings_date TEXT, amount REAL,
    savings_type TEXT,
    FOREIGN KEY(meeting_id) REFERENCES shg_meetings(meeting_id),
    FOREIGN KEY(member_id) REFERENCES members(member_id)
);
CREATE TABLE IF NOT EXISTS shg_loans (
    loan_id TEXT PRIMARY KEY, shg_id TEXT, member_id TEXT,
    loan_date TEXT, loan_amount REAL, category TEXT, purpose TEXT,
    loan_period_months INTEGER, interest_rate_annual REAL,
    installment_amount REAL, repayment_frequency TEXT,
    principal_outstanding REAL, principal_repaid REAL,
    principal_overdue REAL, interest_overdue REAL,
    status TEXT,
    FOREIGN KEY(shg_id) REFERENCES shgs(shg_id),
    FOREIGN KEY(member_id) REFERENCES members(member_id)
);
CREATE TABLE IF NOT EXISTS loan_repayments (
    repayment_id TEXT PRIMARY KEY, loan_id TEXT, meeting_id TEXT,
    member_id TEXT, repayment_date TEXT,
    principal_paid REAL, interest_paid REAL,
    mode_of_receipt TEXT,
    FOREIGN KEY(loan_id) REFERENCES shg_loans(loan_id),
    FOREIGN KEY(member_id) REFERENCES members(member_id)
);
CREATE TABLE IF NOT EXISTS member_other_receipts (
    receipt_id TEXT PRIMARY KEY, meeting_id TEXT, shg_id TEXT,
    member_id TEXT, receipt_date TEXT, receipt_type TEXT,
    amount REAL, mode_of_receipt TEXT,
    FOREIGN KEY(meeting_id) REFERENCES shg_meetings(meeting_id),
    FOREIGN KEY(member_id) REFERENCES members(member_id)
);

-- CLF TRANSACTIONS
CREATE TABLE IF NOT EXISTS clf_loans_to_vo (
    clf_loan_id TEXT PRIMARY KEY, clf_id TEXT, vo_id TEXT,
    loan_date TEXT, fund_type TEXT, loan_amount REAL,
    interest_rate REAL, loan_period_months INTEGER,
    amount_repaid REAL, outstanding REAL, status TEXT,
    FOREIGN KEY(clf_id) REFERENCES clfs(clf_id),
    FOREIGN KEY(vo_id) REFERENCES vos(vo_id)
);
CREATE TABLE IF NOT EXISTS clf_loans_to_shg (
    clf_loan_id TEXT PRIMARY KEY, clf_id TEXT, vo_id TEXT, shg_id TEXT,
    loan_date TEXT, fund_type TEXT, loan_amount REAL,
    interest_rate REAL, loan_period_months INTEGER,
    amount_repaid REAL, outstanding REAL, status TEXT,
    FOREIGN KEY(clf_id) REFERENCES clfs(clf_id),
    FOREIGN KEY(shg_id) REFERENCES shgs(shg_id)
);
CREATE TABLE IF NOT EXISTS clf_bank_transactions (
    txn_id TEXT PRIMARY KEY, clf_id TEXT, txn_date TEXT,
    txn_type TEXT, amount REAL, bank_name TEXT,
    description TEXT,
    FOREIGN KEY(clf_id) REFERENCES clfs(clf_id)
);
CREATE TABLE IF NOT EXISTS clf_fixed_deposits (
    fd_id TEXT PRIMARY KEY, clf_id TEXT, investment_date TEXT,
    maturity_date TEXT, bank_name TEXT, ifsc TEXT,
    amount REAL, interest_rate REAL, investment_type TEXT,
    FOREIGN KEY(clf_id) REFERENCES clfs(clf_id)
);

-- DIGITAL AAJEEVIKA REGISTER
CREATE TABLE IF NOT EXISTS aajeevika_register (
    register_id TEXT PRIMARY KEY, member_id TEXT, shg_id TEXT,
    fiscal_year TEXT, quarter TEXT,
    livelihood_activity TEXT, sub_activity TEXT,
    capital_category TEXT, capital_source TEXT, capital_amount REAL,
    total_funding REAL, annual_income REAL,
    is_potential_lakhpati INTEGER,
    FOREIGN KEY(member_id) REFERENCES members(member_id)
);
"""

# ─── Generator class ─────────────────────────────────────────────────────────

class LokOSGenerator:

    def __init__(self, conn):
        self.conn = conn
        self.cur = conn.cursor()
        self.cur.executescript(SCHEMA)
        self.conn.commit()

        # Counters for unique IDs
        self._counters = {}
        self.all_shgs = []       # list of dicts with shg metadata
        self.all_members = []
        self.all_vos = []
        self.all_clfs = []

    def _next(self, key):
        self._counters[key] = self._counters.get(key, 0) + 1
        return self._counters[key]

    def _insert(self, table, row):
        cols = ", ".join(row.keys())
        placeholders = ", ".join("?" for _ in row)
        self.cur.execute(f"INSERT OR IGNORE INTO {table} ({cols}) VALUES ({placeholders})", list(row.values()))

    # ── Geography ─────────────────────────────────────────────────────────────

    def generate_geography(self):
        print("  Generating geography...")
        for s in STATES:
            self._insert("states", {"state_id": s["id"], "state_name": s["name"], "srlm_name": s["srlm"]})

        self.geo = {}  # block_id -> {district_id, state_id}
        district_num = 0
        block_num = 0
        gp_num = 0

        for s in STATES:
            districts = random.sample(DISTRICTS[s["id"]], min(7, len(DISTRICTS[s["id"]])))
            for dname in districts:
                district_num += 1
                did = f"DIST{district_num:03d}"
                self._insert("districts", {"district_id": did, "district_name": dname, "state_id": s["id"]})

                for b in range(BLOCKS_PER_DISTRICT):
                    block_num += 1
                    bid = f"BLK{block_num:04d}"
                    bname = f"{dname} Block {b+1}"
                    self._insert("blocks", {"block_id": bid, "block_name": bname, "district_id": did})

                    # 2–3 GPs per block
                    for g in range(random.randint(2, 3)):
                        gp_num += 1
                        gpid = f"GP{gp_num:05d}"
                        gpname = f"{dname}-GP-{gp_num}"
                        self._insert("gram_panchayats", {"gp_id": gpid, "gp_name": gpname, "block_id": bid})
                        self.geo.setdefault(bid, []).append(gpid)

        self.conn.commit()
        self.all_blocks = list(self.geo.keys())

    # ── CLFs ──────────────────────────────────────────────────────────────────

    def generate_clfs(self):
        print("  Generating CLFs...")
        clf_num = 0
        for bid in self.all_blocks:
            for c in range(random.randint(2, 3)):
                clf_num += 1
                clf_id = f"CLF{clf_num:04d}"
                form_date = rand_date(date(2015, 1, 1), date(2021, 12, 31))
                reg_date = form_date + timedelta(days=random.randint(180, 540))
                bank = random.choice(BANK_NAMES)
                balance = round(random.uniform(50000, 500000), 2)
                row = {
                    "clf_id": clf_id,
                    "clf_name": f"CLF-{bid}-{c+1}",
                    "block_id": bid,
                    "formation_date": date_str(form_date),
                    "registration_number": f"REG{random.randint(10000,99999)}",
                    "registration_act": random.choice(REGISTRATION_ACTS),
                    "date_of_registration": date_str(reg_date),
                    "status": random.choices(["Active","Revived"], weights=[90,10])[0],
                    "meeting_frequency": random.choice(["Monthly","Fortnightly"]),
                    "financial_intermediation": "Yes",
                    "member_organisation": random.choice(["SHG","VO"]),
                    "compulsory_saving_freq": random.choice(["Monthly","Fortnightly"]),
                    "compulsory_saving_amt": random.choice([100, 150, 200, 250]),
                    "compulsory_saving_rate": random.choice([12, 18, 24]),
                    "bookkeeper_name": name(),
                    "bookkeeper_phone": fake_phone(),
                    "bank_name": bank,
                    "bank_account": fake_account(),
                    "ifsc": fake_ifsc(bank),
                    "bank_balance": balance,
                }
                self._insert("clfs", row)
                self.all_clfs.append({"clf_id": clf_id, "block_id": bid, "formation_date": form_date})
        self.conn.commit()
        print(f"    → {len(self.all_clfs)} CLFs")

    # ── VOs ───────────────────────────────────────────────────────────────────

    def generate_vos(self):
        print("  Generating VOs...")
        vo_num = 0
        self.clf_to_vos = {}

        for clf in self.all_clfs:
            bid = clf["block_id"]
            gps = self.geo.get(bid, [])
            n_vos = random.randint(5, 9)
            for v in range(n_vos):
                vo_num += 1
                vo_id = f"VO{vo_num:05d}"
                gpid = random.choice(gps)
                form_date = clf["formation_date"] + timedelta(days=random.randint(30, 365))
                bank = random.choice(BANK_NAMES)
                row = {
                    "vo_id": vo_id,
                    "vo_name": f"VO-{gpid}-{v+1}",
                    "gp_id": gpid,
                    "clf_id": clf["clf_id"],
                    "formation_date": date_str(form_date),
                    "status": random.choices(["Active","Revived"], weights=[88,12])[0],
                    "meeting_frequency": random.choice(["Monthly","Fortnightly"]),
                    "bookkeeper_name": name(),
                    "bookkeeper_phone": fake_phone(),
                    "bank_name": bank,
                    "bank_account": fake_account(),
                    "ifsc": fake_ifsc(bank),
                    "bank_balance": round(random.uniform(10000, 200000), 2),
                }
                self._insert("vos", row)
                self.all_vos.append({"vo_id": vo_id, "clf_id": clf["clf_id"], "formation_date": form_date})
                self.clf_to_vos.setdefault(clf["clf_id"], []).append(vo_id)
        self.conn.commit()
        print(f"    → {len(self.all_vos)} VOs")

    # ── SHGs ──────────────────────────────────────────────────────────────────

    def generate_shgs(self):
        print("  Generating SHGs...")
        shg_num = 0
        self.vo_to_shgs = {}

        for vo in self.all_vos:
            n_shgs = random.randint(5, 9)
            for s in range(n_shgs):
                shg_num += 1
                shg_id = f"SHG{shg_num:06d}"
                form_date = vo["formation_date"] + timedelta(days=random.randint(30, 730))
                if form_date > date(2024, 12, 31):
                    form_date = date(2024, 12, 31)
                bank = random.choice(BANK_NAMES)
                rf = random.choices([0, 15000, 20000, 25000, 30000], weights=[5,30,30,25,10])[0]
                cif = random.choices([0, 50000, 100000, 150000, 200000, 250000], weights=[10,20,25,20,15,10])[0]
                corpus = round(rf + cif + random.uniform(10000, 80000), 2)
                row = {
                    "shg_id": shg_id,
                    "shg_name": f"SHG-{vo['vo_id']}-{s+1}",
                    "vo_id": vo["vo_id"],
                    "formation_date": date_str(form_date),
                    "status": random.choices(["Active","New","Revived"], weights=[80,12,8])[0],
                    "promoted_by": random.choices(["NRLM","State Project"], weights=[80,20])[0],
                    "meeting_frequency": random.choices(["Weekly","Fortnightly","Monthly"], weights=[55,25,20])[0],
                    "compulsory_saving_amt": random.choice([50, 100, 150, 200]),
                    "compulsory_saving_rate": random.choice([12, 18, 24]),
                    "voluntary_saving": random.choice(["Yes","No"]),
                    "bookkeeper_name": name(),
                    "bookkeeper_phone": fake_phone(),
                    "bank_name": bank,
                    "bank_account": fake_account(),
                    "ifsc": fake_ifsc(bank),
                    "bank_balance": round(random.uniform(5000, 150000), 2),
                    "revolving_fund_recd": rf,
                    "cif_recd": cif,
                    "total_corpus": corpus,
                    "panchsutra_score": random.randint(2, 5),
                }
                self._insert("shgs", row)
                self.all_shgs.append({
                    "shg_id": shg_id, "vo_id": vo["vo_id"],
                    "formation_date": form_date,
                    "saving_amt": row["compulsory_saving_amt"],
                    "meeting_freq": row["meeting_frequency"],
                    "corpus": corpus,
                })
                self.vo_to_shgs.setdefault(vo["vo_id"], []).append(shg_id)
        self.conn.commit()
        print(f"    → {len(self.all_shgs)} SHGs")

    # ── Members ───────────────────────────────────────────────────────────────

    def generate_members(self):
        print("  Generating members...")
        member_num = 0
        self.shg_to_members = {}
        used_aadhaar = set()

        for shg in self.all_shgs:
            n = random.randint(*MEMBERS_PER_SHG)
            roles = ["President","Secretary","Treasurer"] + [None]*(n-3)
            random.shuffle(roles)
            for m in range(n):
                member_num += 1
                mid = f"MBR{member_num:08d}"
                while True:
                    aad = fake_aadhaar()
                    if aad not in used_aadhaar:
                        used_aadhaar.add(aad)
                        break
                dob = rand_date(date(1970, 1, 1), date(2000, 12, 31))
                jdate = shg["formation_date"] + timedelta(days=random.randint(0, 180))
                row = {
                    "member_id": mid,
                    "shg_id": shg["shg_id"],
                    "member_name": name(),
                    "aadhaar": aad,
                    "mobile": fake_phone() if random.random() > 0.2 else None,
                    "dob": date_str(dob),
                    "caste_category": random.choices(["SC","ST","OBC","General"], weights=[20,35,30,15])[0],
                    "bpl_status": random.choices(["BPL","APL"], weights=[70,30])[0],
                    "is_lakhpati_didi": 1 if random.random() < 0.12 else 0,
                    "joining_date": date_str(jdate),
                    "is_office_bearer": 1 if roles[m] else 0,
                    "office_bearer_role": roles[m],
                }
                self._insert("members", row)
                self.shg_to_members.setdefault(shg["shg_id"], []).append({
                    "member_id": mid, "is_lakhpati": row["is_lakhpati_didi"]
                })
                self.all_members.append({"member_id": mid, "shg_id": shg["shg_id"]})
        self.conn.commit()
        print(f"    → {len(self.all_members)} members")

    # ── SHG Meetings & Savings ────────────────────────────────────────────────

    def generate_shg_transactions(self):
        print("  Generating SHG meetings, savings, loans...")
        meeting_num = 0
        savings_num = 0
        loan_num = 0
        repayment_num = 0
        receipt_num = 0

        TODAY = date(2025, 3, 1)

        for shg in self.all_shgs:
            shg_id = shg["shg_id"]
            members = self.shg_to_members.get(shg_id, [])
            if not members:
                continue

            # determine meeting dates over past 2 years
            freq = shg["meeting_freq"]
            if freq == "Weekly":
                interval = 7
            elif freq == "Fortnightly":
                interval = 14
            else:
                interval = 30

            start = max(shg["formation_date"] + timedelta(days=30), date(2023, 4, 1))
            meeting_dates = []
            d = start
            while d <= TODAY:
                meeting_dates.append(d)
                d += timedelta(days=interval)
            meeting_dates = meeting_dates[-48:]  # cap at 48 meetings

            # Track active loans for repayments
            active_loans = []

            cash_in_hand = 0.0
            for mdate in meeting_dates:
                meeting_num += 1
                mid = f"MTG{meeting_num:08d}"

                # Attendance
                present = random.randint(max(1, len(members)-3), len(members))
                present_members = random.sample(members, present)

                # Savings
                total_savings = 0.0
                for mbr in present_members:
                    savings_num += 1
                    amt = shg["saving_amt"] * random.uniform(0.9, 1.1)
                    amt = round(amt, 0)
                    total_savings += amt
                    self._insert("member_savings", {
                        "savings_id": f"SAV{savings_num:09d}",
                        "meeting_id": mid, "shg_id": shg_id,
                        "member_id": mbr["member_id"],
                        "savings_date": date_str(mdate),
                        "amount": amt,
                        "savings_type": "Compulsory",
                    })

                # Loan repayments
                total_repayments = 0.0
                still_active = []
                for loan in active_loans:
                    if loan["outstanding"] <= 0:
                        continue
                    # ~90% chance they repay this meeting
                    if random.random() < 0.90:
                        repayment_num += 1
                        pay_principal = min(loan["installment"], loan["outstanding"])
                        pay_interest = round(loan["outstanding"] * loan["rate_monthly"], 2)
                        loan["outstanding"] = round(loan["outstanding"] - pay_principal, 2)
                        loan["repaid"] = round(loan["repaid"] + pay_principal, 2)
                        total_repayments += pay_principal + pay_interest
                        self._insert("loan_repayments", {
                            "repayment_id": f"REP{repayment_num:09d}",
                            "loan_id": loan["loan_id"],
                            "meeting_id": mid,
                            "member_id": loan["member_id"],
                            "repayment_date": date_str(mdate),
                            "principal_paid": pay_principal,
                            "interest_paid": pay_interest,
                            "mode_of_receipt": random.choice(MODES),
                        })
                    if loan["outstanding"] > 0:
                        still_active.append(loan)
                active_loans = still_active

                # New loan? (~25% chance per meeting if corpus allows)
                total_loans_disbursed = 0.0
                if random.random() < 0.25 and members:
                    mbr = random.choice(members)
                    category = random.choice(LOAN_CATEGORIES)
                    max_loan = min(shg["corpus"] * 0.15, 30000)
                    amt = round(random.uniform(2000, max(2001, max_loan)), -2)
                    period = random.choice([6, 9, 12, 18, 24])
                    rate_annual = random.choice([18, 24, 24, 24, 30])
                    rate_monthly = rate_annual / 1200
                    installment = round((amt * rate_monthly) / (1 - (1 + rate_monthly)**(-period)), 2)
                    loan_num += 1
                    loan_id = f"LN{loan_num:08d}"
                    self._insert("shg_loans", {
                        "loan_id": loan_id, "shg_id": shg_id,
                        "member_id": mbr["member_id"],
                        "loan_date": date_str(mdate),
                        "loan_amount": amt, "category": category,
                        "purpose": random.choice(LOAN_PURPOSES[category]),
                        "loan_period_months": period,
                        "interest_rate_annual": rate_annual,
                        "installment_amount": installment,
                        "repayment_frequency": "Monthly",
                        "principal_outstanding": amt,
                        "principal_repaid": 0, "principal_overdue": 0,
                        "interest_overdue": 0,
                        "status": "Active",
                    })
                    active_loans.append({
                        "loan_id": loan_id,
                        "member_id": mbr["member_id"],
                        "outstanding": amt, "repaid": 0.0,
                        "installment": amt / period,
                        "rate_monthly": rate_monthly,
                    })
                    total_loans_disbursed = amt

                # Occasional other receipts
                if random.random() < 0.15:
                    mbr = random.choice(present_members)
                    receipt_num += 1
                    self._insert("member_other_receipts", {
                        "receipt_id": f"RCP{receipt_num:09d}",
                        "meeting_id": mid, "shg_id": shg_id,
                        "member_id": mbr["member_id"],
                        "receipt_date": date_str(mdate),
                        "receipt_type": random.choice(RECEIPT_TYPES),
                        "amount": round(random.uniform(50, 500), 2),
                        "mode_of_receipt": random.choice(MODES),
                    })

                cash_in_hand = max(0, cash_in_hand + total_savings + total_repayments - total_loans_disbursed)

                self._insert("shg_meetings", {
                    "meeting_id": mid, "shg_id": shg_id,
                    "meeting_date": date_str(mdate),
                    "total_members": len(members),
                    "members_present": present,
                    "total_savings_collected": round(total_savings, 2),
                    "total_loan_repayments": round(total_repayments, 2),
                    "total_loans_disbursed": round(total_loans_disbursed, 2),
                    "cash_in_hand": round(cash_in_hand, 2),
                })

            # Update loan outstanding values in shg_loans table
            for loan in active_loans:
                overdue = round(loan["outstanding"] * 0.05, 2) if random.random() < 0.05 else 0.0
                self.cur.execute(
                    "UPDATE shg_loans SET principal_outstanding=?, principal_repaid=?, principal_overdue=?, status=? WHERE loan_id=?",
                    (loan["outstanding"], loan["repaid"], overdue,
                     "Closed" if loan["outstanding"] <= 0 else "Active", loan["loan_id"])
                )

        self.conn.commit()
        print(f"    → {meeting_num} meetings | {savings_num} savings | {loan_num} loans | {repayment_num} repayments")

    # ── CLF Transactions ──────────────────────────────────────────────────────

    def generate_clf_transactions(self):
        print("  Generating CLF transactions...")
        loan_num = 0
        txn_num = 0
        fd_num = 0

        for clf in self.all_clfs:
            clf_id = clf["clf_id"]
            vos = self.clf_to_vos.get(clf_id, [])
            if not vos:
                continue

            clf_form = clf["formation_date"]

            # Loans to VOs
            for vo_id in random.sample(vos, min(len(vos), random.randint(2, 4))):
                loan_num += 1
                lid = f"CLFL{loan_num:06d}"
                ldate = rand_date(clf_form + timedelta(days=365), date(2024, 6, 30))
                fund = random.choice(FUND_TYPES)
                amt = round(random.uniform(50000, 300000), -3)
                period = random.choice([12, 18, 24, 36])
                repaid = round(amt * random.uniform(0.3, 0.9), -2)
                self._insert("clf_loans_to_vo", {
                    "clf_loan_id": lid, "clf_id": clf_id, "vo_id": vo_id,
                    "loan_date": date_str(ldate),
                    "fund_type": fund, "loan_amount": amt,
                    "interest_rate": random.choice([0, 6, 9, 12]),
                    "loan_period_months": period,
                    "amount_repaid": repaid,
                    "outstanding": round(amt - repaid, 2),
                    "status": "Active" if amt - repaid > 0 else "Closed",
                })

            # Loans to SHGs (via VO)
            all_shg_ids = []
            for vo_id in vos:
                all_shg_ids.extend(self.vo_to_shgs.get(vo_id, []))
            sample_shgs = random.sample(all_shg_ids, min(len(all_shg_ids), random.randint(3, 8)))

            for shg_id in sample_shgs:
                loan_num += 1
                lid = f"CLFL{loan_num:06d}"
                vo_id = next((v["vo_id"] for v in self.all_vos if v["vo_id"] in vos), vos[0])
                ldate = rand_date(clf_form + timedelta(days=180), date(2024, 9, 30))
                amt = round(random.uniform(20000, 150000), -3)
                repaid = round(amt * random.uniform(0.2, 0.95), -2)
                self._insert("clf_loans_to_shg", {
                    "clf_loan_id": lid, "clf_id": clf_id, "vo_id": vo_id,
                    "shg_id": shg_id,
                    "loan_date": date_str(ldate),
                    "fund_type": random.choice(FUND_TYPES),
                    "loan_amount": amt,
                    "interest_rate": random.choice([0, 6, 9]),
                    "loan_period_months": random.choice([12, 18, 24]),
                    "amount_repaid": repaid,
                    "outstanding": round(amt - repaid, 2),
                    "status": "Active" if amt - repaid > 0 else "Closed",
                })

            # Bank transactions
            for _ in range(random.randint(6, 15)):
                txn_num += 1
                tdate = rand_date(clf_form + timedelta(days=60), date(2025, 2, 28))
                txn_type = random.choice(["CIF Receipt","RF Receipt","Repayment Received","Operating Expense","Member Savings Deposit"])
                self._insert("clf_bank_transactions", {
                    "txn_id": f"CTXN{txn_num:07d}",
                    "clf_id": clf_id,
                    "txn_date": date_str(tdate),
                    "txn_type": txn_type,
                    "amount": round(random.uniform(5000, 500000), 2),
                    "bank_name": random.choice(BANK_NAMES),
                    "description": f"{txn_type} - auto generated",
                })

            # Fixed deposits
            if random.random() < 0.4:
                fd_num += 1
                inv_date = rand_date(clf_form + timedelta(days=365), date(2024, 1, 1))
                mat_date = inv_date + timedelta(days=365)
                bank = random.choice(BANK_NAMES)
                self._insert("clf_fixed_deposits", {
                    "fd_id": f"FD{fd_num:05d}",
                    "clf_id": clf_id,
                    "investment_date": date_str(inv_date),
                    "maturity_date": date_str(mat_date),
                    "bank_name": bank,
                    "ifsc": fake_ifsc(bank),
                    "amount": round(random.uniform(25000, 200000), -3),
                    "interest_rate": random.choice([6.5, 7.0, 7.25, 7.5]),
                    "investment_type": random.choice(["Fixed Deposit","Recurring Deposit"]),
                })

        self.conn.commit()
        print(f"    → {loan_num} CLF loans | {txn_num} bank txns | {fd_num} FDs")

    # ── Digital Aajeevika Register ────────────────────────────────────────────

    def generate_aajeevika(self):
        print("  Generating Digital Aajeevika Register...")
        reg_num = 0
        fiscal_years = ["2022-23", "2023-24", "2024-25"]
        quarters = ["Q1", "Q2", "Q3", "Q4"]
        capital_sources = ["Internal SHG Loan","Bank Linkage","CIF","VRF","Government Scheme","Own Savings","No Capital"]

        # Only ~60% of members have an entry (data entry compliance)
        sample = random.sample(self.all_members, int(len(self.all_members) * 0.60))

        for mbr in sample:
            # 1–3 entries per member across years/quarters
            n_entries = random.randint(1, 3)
            used_periods = set()
            for _ in range(n_entries):
                fy = random.choice(fiscal_years)
                q = random.choice(quarters)
                period_key = (mbr["member_id"], fy, q)
                if period_key in used_periods:
                    continue
                used_periods.add(period_key)

                reg_num += 1
                activity = random.choice(LIVELIHOOD_ACTIVITIES)
                cap_source = random.choice(capital_sources)
                cap_amt = 0.0 if cap_source == "No Capital" else round(random.uniform(2000, 50000), -2)
                annual_income = round(random.uniform(30000, 200000), -2)
                is_lakhpati = 1 if annual_income >= 100000 else 0

                self._insert("aajeevika_register", {
                    "register_id": f"AAJ{reg_num:09d}",
                    "member_id": mbr["member_id"],
                    "shg_id": mbr["shg_id"],
                    "fiscal_year": fy,
                    "quarter": q,
                    "livelihood_activity": activity,
                    "sub_activity": random.choice(LOAN_PURPOSES.get(
                        "Agriculture" if "Farm" in activity or "Crop" in activity
                        else ("NTFP" if "NTFP" in activity else "Non-Farm"), ["General"]
                    )),
                    "capital_category": random.choice(["Loan","Funds","Others","None"]),
                    "capital_source": cap_source,
                    "capital_amount": cap_amt,
                    "total_funding": cap_amt,
                    "annual_income": annual_income,
                    "is_potential_lakhpati": is_lakhpati,
                })
        self.conn.commit()
        print(f"    → {reg_num} Aajeevika Register entries")

    # ── Summary ───────────────────────────────────────────────────────────────

    def print_summary(self):
        tables = [
            "states","districts","blocks","gram_panchayats",
            "clfs","vos","shgs","members",
            "shg_meetings","member_savings","shg_loans","loan_repayments","member_other_receipts",
            "clf_loans_to_vo","clf_loans_to_shg","clf_bank_transactions","clf_fixed_deposits",
            "aajeevika_register"
        ]
        print("\n  ── Row counts ──────────────────────────")
        total = 0
        for t in tables:
            self.cur.execute(f"SELECT COUNT(*) FROM {t}")
            n = self.cur.fetchone()[0]
            total += n
            print(f"    {t:<35} {n:>8,}")
        print(f"    {'TOTAL':<35} {total:>8,}")


# ─── Main ─────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)

    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA synchronous=NORMAL")

    print("🔧 LokOS Synthetic Database Generator")
    print("=" * 45)

    gen = LokOSGenerator(conn)

    print("\n[1/8] Geography")
    gen.generate_geography()

    print("\n[2/8] CLFs")
    gen.generate_clfs()

    print("\n[3/8] VOs")
    gen.generate_vos()

    print("\n[4/8] SHGs")
    gen.generate_shgs()

    print("\n[5/8] Members")
    gen.generate_members()

    print("\n[6/8] SHG Transactions")
    gen.generate_shg_transactions()

    print("\n[7/8] CLF Transactions")
    gen.generate_clf_transactions()

    print("\n[8/8] Digital Aajeevika Register")
    gen.generate_aajeevika()

    print("\n✅ Generation complete!")
    gen.print_summary()

    # Create useful indexes
    print("\n  Creating indexes...")
    conn.executescript("""
        CREATE INDEX IF NOT EXISTS idx_members_shg ON members(shg_id);
        CREATE INDEX IF NOT EXISTS idx_shgs_vo ON shgs(vo_id);
        CREATE INDEX IF NOT EXISTS idx_vos_clf ON vos(clf_id);
        CREATE INDEX IF NOT EXISTS idx_meetings_shg ON shg_meetings(shg_id);
        CREATE INDEX IF NOT EXISTS idx_savings_member ON member_savings(member_id);
        CREATE INDEX IF NOT EXISTS idx_loans_member ON shg_loans(member_id);
        CREATE INDEX IF NOT EXISTS idx_loans_shg ON shg_loans(shg_id);
        CREATE INDEX IF NOT EXISTS idx_repayments_loan ON loan_repayments(loan_id);
        CREATE INDEX IF NOT EXISTS idx_aajeevika_member ON aajeevika_register(member_id);
        CREATE INDEX IF NOT EXISTS idx_aajeevika_shg ON aajeevika_register(shg_id);
    """)
    conn.commit()
    conn.close()

    size_mb = os.path.getsize(DB_PATH) / (1024*1024)
    print(f"\n  Database saved: {DB_PATH}")
    print(f"  File size: {size_mb:.1f} MB")